# 08 — LiDAR, Hillshades, and Landslide Recognition

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Katowice &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

Airborne LiDAR has revolutionised landslide mapping. A 1-m bare-earth DTM removes vegetation, exposes subtle morphology, and reveals dormant and historical landslides that no field campaign and no aerial photograph could find. Poland's [SOPO inventory](https://geoportal.gov.pl) and Norway's [hoydedata.no](https://hoydedata.no) portal are two of the best public examples — both built on national LiDAR coverage.

This notebook covers the **four core operations** every landslide mapper needs to do with a DEM: read it, hillshade it under multiple sun angles, compute slope and aspect, and recognise the morphological signature of a slide. We do this on a synthetic Carpathian-style DEM with two landslide features baked in. The notebook is built so you can swap in a real tile (ISOK 1-m, hoydedata 10-m, EU-DEM 25-m) by setting a single path.


## About this notebook

**Learning objectives.** By the end of this notebook the student will be able to:

1. Read a DEM, compute its hillshade under arbitrary sun azimuth and altitude, and explain why multiple sun angles are needed for landslide mapping.
2. Compute slope and aspect from a DEM and interpret the maps geomorphologically.
3. Recognise the canonical morphological signature of a translational/rotational landslide on a hillshade: arcuate headscarp, hummocky deposit, lateral shear margins.
4. Swap a real LiDAR-derived DTM into the workflow by changing one variable.

**Prerequisites.** Notebook 04 (infinite slope) for the connection between slope angle and stability. The rest is self-contained.

**On data.** This notebook ships with a fully reproducible synthetic DEM that includes two embedded landslide features. To use a real tile (Polish ISOK, Norwegian høydedata, EU-DEM, OpenTopography), set `DEM_PATH` in §2. The `rasterio` library is the standard reader for GeoTIFF; it is commented out in `requirements.txt` and can be enabled when you start using real data.

> **For your PowerPoint deck.** Four SVG figures land in `figures/`:
> - `dem_elevation.svg` — coloured elevation map with contours
> - `hillshade_sun_angles.svg` — four hillshades from four sun azimuths (the classical "which scarp shows up?" figure)
> - `slope_aspect.svg` — derived slope and aspect maps
> - `landslide_close_up.svg` — zoom on a landslide with morphology annotated


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from scipy.ndimage import gaussian_filter

from style import apply_style, COLORS, save_figure
apply_style()


## 1. Why LiDAR for landslides

Three things that LiDAR-derived bare-earth DTMs do that no other data source does:

- **See through the canopy.** In the forested Carpathians, Sudetes, and Norwegian coastal slopes, vegetation hides 60–90 % of pre-existing landslide morphology. LiDAR's last-return filtering produces a bare-earth model where dormant slides are unmistakable.
- **Resolve metre-scale features.** A 0.5–1 m DTM resolves individual scarps, hummocks, and channel margins. Photogrammetric DEMs at 5–10 m resolution average these out.
- **Cover whole landscapes.** Once a national coverage exists, mapping is a desk job. Poland's SOPO inventory and the Norwegian Geological Survey's landslide map both jumped in completeness by an order of magnitude after national LiDAR was flown.

The flip side is data volume: a 1-m DTM of a 10 × 10 km area is ~200 MB, and access to high-resolution national datasets is sometimes politically or commercially restricted. Most teaching workflows therefore start with a coarser DEM (10–25 m) and zoom into 1 m only over the specific area of interest.


## 2. The DEM — synthetic or real

The cell below builds a synthetic 1000 × 600 m Carpathian-style hillslope at 5 m resolution, with two pre-embedded translational landslides for teaching. If you have a real DTM tile (GeoTIFF), set `DEM_PATH` to its filesystem path and it will be loaded instead.


In [ ]:
# ---- SWAP-IN POINT FOR A REAL DEM ----
# Set to a string path to a GeoTIFF to use a real tile.
# Requires `pip install rasterio` and uncommenting it in requirements.txt.
DEM_PATH = None
# DEM_PATH = "/path/to/your/dem_tile.tif"
# DEM_PATH = "/path/to/sopo_warsaw.tif"

# Resolution in metres along x and y. For a real DTM this is read from the file.
dx_default = 5.0


def add_landslide(z, cx, cy, length, width, depth, dx, hummock_seed=0):
    """Add an arcuate scarp at (cx, cy) and a hummocky deposit east of it."""
    ny, nx = z.shape
    yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing="ij")
    rng = np.random.default_rng(hummock_seed)

    # Arcuate headscarp: depression on/west of the centre
    r = np.hypot(xx - cx, yy - cy)
    in_scar = (r < width) & (xx <= cx + 2)
    z = z - depth * np.exp(-(r / width) ** 2) * in_scar

    # Deposit: raised, hummocky lobe to the east
    dx_local = xx - cx
    dy_local = yy - cy
    in_dep = (dx_local > 0) & (dx_local < length) & (np.abs(dy_local) < width * 1.1)
    envelope = np.exp(
        -((dx_local - length / 2) ** 2 + dy_local ** 2) / (width * length / 3)
    )
    hummock_noise = gaussian_filter(rng.standard_normal(z.shape), sigma=2.0)
    hummocks = 1.0 + 0.5 * hummock_noise / max(abs(hummock_noise).max(), 1e-9)
    deposit = depth * 0.35 * envelope * hummocks
    z = z + np.where(in_dep, deposit, 0.0)

    return z


def make_synthetic_dem(seed=42, shape=(120, 200), dx=5.0):
    """Synthetic Carpathian-style south-east-facing slope with two landslides."""
    rng = np.random.default_rng(seed)
    ny, nx = shape
    yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing="ij")

    # Base tilt: west high (1100 m), east low (~950 m at x = 1000 m)
    z = 1100.0 - 0.15 * xx * dx

    # Across-slope valley structure (gentle channels running E-W)
    z += 18.0 * np.sin(0.018 * yy * dx + 1.3) * (1.0 - 0.3 * xx / nx)

    # Multi-scale noise: medium ridges + fine texture
    noise_med = gaussian_filter(rng.standard_normal(shape), sigma=8)
    z += 22.0 * (noise_med - noise_med.mean()) / (noise_med.std() + 1e-9)
    noise_small = gaussian_filter(rng.standard_normal(shape), sigma=1.5)
    z += 2.0 * (noise_small - noise_small.mean()) / (noise_small.std() + 1e-9)

    # Two embedded landslides
    z = add_landslide(z, cx=55,  cy=40, length=28, width=16, depth=14, dx=dx, hummock_seed=1)
    z = add_landslide(z, cx=120, cy=80, length=42, width=24, depth=22, dx=dx, hummock_seed=2)
    return z


if DEM_PATH is not None:
    try:
        import rasterio
        with rasterio.open(DEM_PATH) as src:
            dem = src.read(1, masked=True).filled(np.nan).astype(float)
            xres = abs(src.transform.a); yres = abs(src.transform.e)
        dx, dy = xres, yres
        print(f"Loaded real DEM:  shape = {dem.shape},  dx = {dx:.2f} m")
    except ImportError:
        print("rasterio not installed.  pip install rasterio  inside your venv.")
        print("Falling back to synthetic DEM.")
        dem = make_synthetic_dem()
        dx = dy = dx_default
else:
    dem = make_synthetic_dem()
    dx = dy = dx_default
    print(f"Using synthetic DEM:  shape = {dem.shape},  dx = {dx:.2f} m")
    print(f"Elevation range: {dem.min():.1f} - {dem.max():.1f} m a.s.l.")

# Geographical extent for plotting (metres from SW corner)
extent = (0.0, dem.shape[1] * dx, 0.0, dem.shape[0] * dy)


In [ ]:
fig, ax = plt.subplots(figsize=(10.0, 5.5))
im = ax.imshow(dem, extent=extent, origin="upper", cmap="terrain")
cs = ax.contour(np.flipud(dem),
                levels=np.arange(950, 1110, 10),
                extent=extent, colors="black", linewidths=0.5, alpha=0.6)
ax.clabel(cs, fmt="%d", fontsize=8, inline=True)
cb = fig.colorbar(im, ax=ax, shrink=0.92)
cb.set_label("elevation [m a.s.l.]")
ax.set_xlabel("easting [m]")
ax.set_ylabel("northing [m]")
ax.set_title("Synthetic Carpathian-style DEM (with two embedded landslides)")
save_figure(fig, "dem_elevation")
plt.show()


## 3. Hillshades — making topography visible

A **hillshade** is a greyscale rendering of the terrain that simulates illumination from a sun at altitude $h$ (degrees above horizon) and azimuth $A$ (degrees clockwise from north). Each pixel's brightness depends on the angle between its surface normal and the incident light:

$$
I \;=\; \cos\theta_z \cos\beta \;+\; \sin\theta_z \sin\beta\,\cos(A - \alpha),
\qquad (1)
$$

where $\theta_z = 90^\circ - h$ is the solar zenith angle, $\beta$ is the slope, and $\alpha$ is the aspect. The result is clipped to $[0, 1]$ and displayed as greyscale (Horn 1981; this is the ESRI/GDAL convention).

A single hillshade has a critical weakness: **features parallel to the sun direction disappear**. A NW–SE-trending scarp becomes invisible under a NW (315°) sun because the light grazes along it. Standard landslide-mapping practice is therefore to look at the same DEM under **at least four sun azimuths** — typically NE (45°), SE (135°), SW (225°), NW (315°) — at a moderate altitude (30–45°). Anything that appears in all four is real; anything visible only at one angle may be an illumination artefact.


In [ ]:
sun_altitude = 35.0
azimuths = [45, 135, 225, 315]
labels = ["NE  (45°)", "SE  (135°)", "SW  (225°)", "NW  (315°)"]

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.0))
for ax, az, lab in zip(axes.flat, azimuths, labels):
    ls = LightSource(azdeg=az, altdeg=sun_altitude)
    hill = ls.hillshade(dem, vert_exag=2.0, dx=dx, dy=dy)
    ax.imshow(hill, extent=extent, origin="upper", cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"sun {lab},  altitude {sun_altitude:.0f}°")
    ax.set_xlabel("easting [m]"); ax.set_ylabel("northing [m]")
fig.suptitle("Same DEM, four sun azimuths — features parallel to the sun disappear", y=1.02)
save_figure(fig, "hillshade_sun_angles")
plt.show()


## 4. Slope and aspect

Slope $\beta$ and aspect $\alpha$ are the two first-order derivatives of the DEM. Given the gradient components $\partial z/\partial x$ (eastward) and $\partial z/\partial y$ (northward):

$$
\beta \;=\; \arctan\sqrt{\left(\frac{\partial z}{\partial x}\right)^2 + \left(\frac{\partial z}{\partial y}\right)^2},
\qquad (2)
$$

$$
\alpha \;=\; \arctan\!2\!\left(\frac{\partial z}{\partial x},\; -\frac{\partial z}{\partial y}\right) \bmod 360^\circ,
\qquad (3)
$$

where $\alpha$ is the **direction the slope faces** (compass azimuth of the downhill direction). For landslide work, slope is the single most important DEM-derived layer: every shallow translational slide in the Carpathians initiates on slopes of 25–45°. A slope map immediately highlights the **release zones** of potential failures.


In [ ]:
# Slope and aspect via central differences. Note that np.gradient returns
# (dz/dy, dz/dx) because axis 0 is rows (y) and axis 1 is columns (x).
# We invert the y-gradient because image-y increases downward (= south).
dz_dy_img, dz_dx = np.gradient(dem, dy, dx)
dz_dy = -dz_dy_img                            # so dz_dy is northward gradient
slope_deg = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy)))
aspect_deg = (np.degrees(np.arctan2(dz_dx, -dz_dy)) + 360.0) % 360.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5.0))

im1 = ax1.imshow(slope_deg, extent=extent, origin="upper",
                 cmap="YlOrRd", vmin=0, vmax=55)
cb1 = fig.colorbar(im1, ax=ax1, shrink=0.92)
cb1.set_label(r"slope  $\beta$  [$^\circ$]")
ax1.set_xlabel("easting [m]"); ax1.set_ylabel("northing [m]")
ax1.set_title(r"Slope angle  —  highlights potential release zones ($\beta$ > 25°)")
ax1.contour(np.flipud(slope_deg), levels=[25, 35, 45],
            extent=extent, colors="black", linewidths=0.5, alpha=0.6)

im2 = ax2.imshow(aspect_deg, extent=extent, origin="upper",
                 cmap="twilight", vmin=0, vmax=360)
cb2 = fig.colorbar(im2, ax=ax2, shrink=0.92)
cb2.set_label(r"aspect  $\alpha$  [$^\circ$] (compass)")
ax2.set_xlabel("easting [m]"); ax2.set_ylabel("northing [m]")
ax2.set_title("Aspect  —  direction the slope faces")

save_figure(fig, "slope_aspect")
plt.show()

print(f"Slope:  mean = {slope_deg.mean():.1f}°,  max = {slope_deg.max():.1f}°")
print(f"Fraction of pixels above 25°: {(slope_deg > 25).mean() * 100:.1f} %")


## 5. Recognising a landslide on a hillshade

A textbook translational or rotational slide leaves a recognisable signature on a high-resolution hillshade:

- **Arcuate headscarp.** A crescent-shaped escarpment at the top, concave downslope. On hillshade it appears as a sharp shadow line on the uphill (sun-facing) side and a bright highlight on the downhill side.
- **Hummocky deposit.** Below the scarp, the slid mass produces a chaotic mound of disturbed material with metre-scale relief — visible as patches of dark and light in the hillshade.
- **Lateral shear margins.** Two roughly parallel scars at the sides of the slide where it sheared past the surrounding ground.
- **Reverse slopes.** Mini back-tilted blocks within the deposit, characteristic of rotational failures.

The combined hillshade-plus-colour-elevation rendering below makes these features stand out. We zoom on the **larger of the two embedded landslides** (centred at the eastern half of the DEM).


In [ ]:
# Zoom on the larger embedded landslide (cx=120, cy=80, in pixel coordinates).
# Convert to map coordinates for the slice.
x0_px, x1_px = 90, 175
y0_px, y1_px = 50, 115
sub = dem[y0_px:y1_px, x0_px:x1_px]
sub_extent = (x0_px * dx, x1_px * dx, (dem.shape[0] - y1_px) * dy, (dem.shape[0] - y0_px) * dy)

ls = LightSource(azdeg=315, altdeg=40)
rgb = ls.shade(sub, cmap=plt.cm.terrain, blend_mode="soft", vert_exag=2.5,
               dx=dx, dy=dy)

fig, ax = plt.subplots(figsize=(8.5, 6.0))
ax.imshow(rgb, extent=sub_extent, origin="upper")

# Annotate the morphology
sub_xc = (x0_px + (120 - x0_px)) * dx
sub_yc = (dem.shape[0] - 80) * dy  # convert image-y to map-y
ax.annotate("arcuate headscarp",
            xy=(120 * dx, (dem.shape[0] - 70) * dy),
            xytext=(120 * dx - 100, (dem.shape[0] - 50) * dy),
            color=COLORS["fail"], fontsize=12,
            arrowprops=dict(arrowstyle="->", color=COLORS["fail"], lw=1.5))
ax.annotate("hummocky deposit",
            xy=((120 + 25) * dx, (dem.shape[0] - 80) * dy),
            xytext=((120 + 30) * dx, (dem.shape[0] - 110) * dy),
            color=COLORS["accent"], fontsize=12,
            arrowprops=dict(arrowstyle="->", color=COLORS["accent"], lw=1.5))

ax.set_xlabel("easting [m]"); ax.set_ylabel("northing [m]")
ax.set_title("Translational landslide on hillshade-coloured DEM  (sun NW, alt 40°)")
save_figure(fig, "landslide_close_up")
plt.show()


## 6. Using a real LiDAR tile

To replace the synthetic DEM with a real one, two steps:

1. **Get a tile.** Public sources, in order of preference:
   - **Norway:** [hoydedata.no](https://hoydedata.no) — free national 10 m DTM, plus 1 m where available.
   - **Poland:** [geoportal.gov.pl / ISOK](https://geoportal.gov.pl) — national 1 m DTM, free with registration.
   - **Europe-wide:** [Copernicus EU-DEM](https://www.copernicus.eu/en/access-data) — 25 m, no registration.
   - **Global:** [OpenTopography](https://opentopography.org) — aggregates many open datasets, has a clean API.

2. **Wire it in.** Uncomment `rasterio` in `requirements.txt`, install with `pip install rasterio`, then in §2 of this notebook set `DEM_PATH = "your_tile.tif"` and rerun the cells from §2 downward. Everything else in the notebook (hillshade, slope/aspect, close-ups) is DEM-agnostic and will work on any GeoTIFF.

For an in-class demo, a 5–10 MB clip is a good size: large enough to show real morphology, small enough to load fast and ship in a teaching repo if the licence permits.


## 7. Where to go next

This notebook covers the *visualisation* end of LiDAR-based landslide work. The next layers, all built on the same DEM, are operational tools used in regional-scale studies:

- **Curvature and roughness.** Second derivatives of the DEM. Local roughness (standard deviation of elevation in a moving window) is the most reliable single diagnostic of landslide deposits — they are systematically rougher than the surrounding terrain. See Berti et al. (2013), McKean & Roering (2004).
- **Object-based image analysis (OBIA).** Segments the DEM and its derivatives into objects, then classifies them as "slide", "scarp", "deposit", "stable" via random-forest or CNN models. Strong recent results on Polish and Italian LiDAR.
- **Susceptibility mapping.** Combines the slope, aspect, and lithology layers in a statistical (logistic-regression) or physically-based (infinite-slope from notebook 04) framework to predict where future slides are likely. The output is a probability map at landscape scale.
- **Difference-of-DEMs.** Subtract two LiDAR campaigns to detect new failures, erosion, and deposition with cm-scale precision. Operational in Norway since 2018 (national geological survey, NGU).

For doctoral students, the high-leverage next step is to learn **`rasterio`** for I/O, **`xarray`** for multi-DEM analysis, and **`scikit-learn`** for classification. The notebooks in this course intentionally stop at the visualisation stage — once you have a 1-m DTM open in a Jupyter kernel, the rest is software engineering.


## Take-aways

- LiDAR bare-earth DTMs are the most important data source in modern landslide mapping. They expose dormant and historical slides that no other technique can.
- **Hillshade** is the single most useful visualisation. Always inspect under several sun azimuths — features parallel to a single sun direction disappear.
- **Slope** highlights potential release zones (β > 25°); **aspect** is mostly used for climate-related questions (snowmelt, freeze-thaw on north-facing slopes).
- The morphological signature of a textbook slide on hillshade is unmistakable: arcuate headscarp, hummocky deposit, lateral shears. Practice reading them on real tiles — ISOK or hoydedata are excellent starting points.
- Operational landslide mapping today is increasingly automated (OBIA, random-forest, CNN). Visual inspection remains the calibration anchor — every automated workflow trains on a hand-mapped subset.


## Questions for the exam

1. Explain why a single hillshade is insufficient for landslide mapping. Which feature orientations are most likely to be missed under a NW (315°) sun?
2. List the four morphological signatures of a translational landslide on a high-resolution hillshade. Which is most diagnostic — and why?
3. A slope-angle map shows a 30 ha patch above 35°. Under what circumstances would you *not* expect landslide initiation in such a patch?
4. What does aspect tell you about landslide hazard in (a) Central Europe and (b) Norway? Why is the relationship different?


## References

- Horn, B. K. P. (1981). *Hill shading and the reflectance map.* Proceedings of the IEEE, 69(1), 14–47.
- McKean, J. & Roering, J. (2004). *Objective landslide detection and surface morphology mapping using high-resolution airborne laser altimetry.* Geomorphology, 57(3–4), 331–351.
- Van Den Eeckhaut, M., Poesen, J., Verstraeten, G., Vanacker, V., Nyssen, J., Moeyersons, J., van Beek, L. P. H., & Vandekerckhove, L. (2007). *Use of LiDAR-derived images for mapping old landslides under forest.* Earth Surface Processes and Landforms, 32(5), 754–769.
- Berti, M., Corsini, A., & Daehne, A. (2013). *Comparative analysis of surface roughness algorithms for the identification of active landslides.* Geomorphology, 182, 1–18.
- Wieczorek, F. G. (1984). *Preparing a detailed landslide-inventory map for hazard evaluation and reduction.* Bulletin of the Association of Engineering Geologists, 21(3), 337–342.
- Pawlik, Ł., Migoń, P., Owczarek, P., & Kacprzak, A. (2013). *Surface processes and interactions with forest vegetation on a steep mudstone slope, Stołowe Mountains, SW Poland.* Catena, 109, 203–216.
